# S4 — Predicting Ad Click-Through Rate (CTR)

Starter notebook: **EDA → features → baseline → Logistic Regression → LightGBM → evaluation.**

**Before running:** download a sample of the Avazu CTR dataset and put `train.csv` in the `data/` folder.
https://www.kaggle.com/competitions/avazu-ctr-prediction/data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix, RocCurveDisplay

pd.set_option('display.max_columns', 50)
sns.set_theme()

## 1. Load the data

The full Avazu file is huge (~40M rows). We read only the first N rows so it loads fast on a laptop.

In [ ]:
N_ROWS = 200_000   # increase later if your machine can handle it

df = pd.read_csv('../data/train.csv', nrows=N_ROWS)
print(df.shape)
df.head()

## 2. Explore — the imbalance story

The single most important slide: **most impressions are NOT clicked.** This is why accuracy is a trap.

In [ ]:
click_rate = df['click'].mean()
print(f'Overall click rate: {click_rate:.3f}  ({click_rate*100:.1f}%)')
print(f'A "never click" model would be {(1-click_rate)*100:.1f}% accurate — but useless.')

df['click'].value_counts().plot(kind='bar', title='Click (1) vs No-click (0)')
plt.show()

In [ ]:
# Time feature: the 'hour' column is formatted YYMMDDHH. Pull out the hour of day.
df['hour_of_day'] = (df['hour'] % 100).astype(int)

ctr_by_hour = df.groupby('hour_of_day')['click'].mean()
ctr_by_hour.plot(marker='o', title='Click rate by hour of day')
plt.ylabel('click rate')
plt.show()

In [ ]:
# How many unique categories does each column have? (high-cardinality columns are tricky to encode)
df.nunique().sort_values(ascending=False)

## 3. Prepare features

We keep a small set of **low-cardinality** categorical columns for a clean first model.
(High-cardinality columns like `device_ip` need target/frequency encoding — a good "next step".)

In [ ]:
cat_features = [
    'C1', 'banner_pos', 'site_category', 'app_category',
    'device_type', 'device_conn_type', 'hour_of_day',
]

X = df[cat_features].astype(str)   # treat all as categorical
y = df['click']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('train:', X_train.shape, ' test:', X_test.shape)

## 4. Baseline

Always predict the average click rate. This is the number every real model must beat.

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())
print(f'Baseline log-loss: {log_loss(y_test, baseline_pred):.4f}')
print('Baseline AUC: 0.500 (a constant prediction cannot rank — AUC is 0.5 by definition)')

## 5. Logistic Regression

One-hot encode the categories, then fit. `class_weight='balanced'` helps with the imbalance.

In [ ]:
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

logreg = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
logreg.fit(X_train, y_train)

proba_lr = logreg.predict_proba(X_test)[:, 1]
print(f'LogReg AUC:      {roc_auc_score(y_test, proba_lr):.4f}')
print(f'LogReg log-loss: {log_loss(y_test, proba_lr):.4f}')

## 6. LightGBM (usually the winner)

Gradient boosting handles categories natively and captures interactions between features.

In [ ]:
import lightgbm as lgb

# LightGBM can use pandas 'category' dtype directly — no one-hot needed.
X_train_lgb = X_train.astype('category')
X_test_lgb = X_test.astype('category')

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
)
model.fit(X_train_lgb, y_train)

proba_lgb = model.predict_proba(X_test_lgb)[:, 1]
print(f'LightGBM AUC:      {roc_auc_score(y_test, proba_lgb):.4f}')
print(f'LightGBM log-loss: {log_loss(y_test, proba_lgb):.4f}')

## 7. Evaluate & visualize (your slide material)

In [ ]:
# ROC curves — baseline vs logistic vs boosting
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_test, proba_lr, name='LogReg', ax=ax)
RocCurveDisplay.from_predictions(y_test, proba_lgb, name='LightGBM', ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Baseline (0.5)')
ax.set_title('ROC Curve')
ax.legend()
plt.show()

In [ ]:
# Feature importance — the 'what drives clicks' slide
importances = pd.Series(model.feature_importances_, index=cat_features).sort_values()
importances.plot(kind='barh', title='LightGBM feature importance')
plt.show()

## Next steps
- Add high-cardinality features (`site_id`, `device_model`) with frequency/target encoding
- Tune LightGBM hyperparameters
- Split train/test **by time** instead of randomly (more realistic)
- Try a calibration curve to check probability quality